In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API keys are loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment")
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY not found in environment")

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [4]:
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI
from tavily import TavilyClient

model = ChatOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"],
    model="gemini-3.5-flash",
    temperature=0.1
)


In [5]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def internet_search(query: str, max_results: int = 5) -> dict:
    """
    Search the web for information using Tavily.
    
    Args:
        query: The search query string
        max_results: Maximum number of results to return (default: 5)
    
    Returns:
        Dictionary containing search results with titles, URLs, and content
    """
    try:
        results = tavily_client.search(query, max_results=max_results)
        return results
    except Exception as e:
        return {"error": f"Search failed: {str(e)}"}

print("Tavily search tool configured successfully!")

Tavily search tool configured successfully!


In [6]:
from pathlib import Path
from deepagents.backends import FilesystemBackend

# Create a dedicated workspace directory for agent files
agent_workspace = Path("./agent_workspace")
agent_workspace.mkdir(exist_ok=True)

print(f"Agent workspace created at: {agent_workspace.absolute()}")

Agent workspace created at: /Users/01598825/workspace/AI_STUDY_DEMO_2/ai-agents/agent_workspace


In [7]:
# Define a detailed system prompt (characteristic #1 of deep agents)
system_prompt = """You are an expert research assistant with access to web search and file management tools.

YOUR ROLE:
- Conduct thorough research on user queries
- Break down complex questions into manageable subtasks
- Synthesize information from multiple sources
- Save important findings to files for reference.

TOOL USAGE GUIDELINES:
- Use internet_search for current information and facts
- Start with broad searches, then narrow based on results
- Always cite sources when presenting information
- Save detailed research findings to files for later reference

WORKFLOW FOR COMPLEX TASKS:
1. Break the task into subtasks using the write_todos tool
2. Execute each subtask systematically
3. Save intermediate findings to files
4. Synthesize results into a comprehensive answer

ERROR HANDLING:
- If a search returns no results, try alternative keywords
- If information is unclear, search for clarification
- Always verify facts from multiple sources when possible
"""

# Create the deep agent with persistent filesystem
agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt=system_prompt,
    backend=FilesystemBackend(root_dir=str(agent_workspace), virtual_mode=True)
)

print("Deep agent created successfully!")
print("\\nAgent capabilities:")
print("- Planning tool (write_todos)")
print("- File system tools (read, write, edit files)")
print("- Internet search via Tavily")
print("- Subagent spawning (task tool)")
print(f"- Files will be saved to: {agent_workspace.absolute()}")

Deep agent created successfully!
\nAgent capabilities:
- Planning tool (write_todos)
- File system tools (read, write, edit files)
- Internet search via Tavily
- Subagent spawning (task tool)
- Files will be saved to: /Users/01598825/workspace/AI_STUDY_DEMO_2/ai-agents/agent_workspace


In [8]:
def stream_agent_response(agent, query, show_details=False, max_content_length=1000):
    """
    Stream and display agent responses with clean formatting.
    
    Args:
        agent: The deep agent instance
        query: The query string to send to the agent
        show_details: If True, show detailed tool arguments and planning
        max_content_length: Maximum characters to show for content (None for full content)
    """
    print(f"Query: {query if len(query) <= 100 else query[:100] + '...'}")
    print("\n" + "="*80)
    print("Streaming Agent Actions:")
    print("="*80 + "\n")
    
    # Track message IDs to avoid duplicates
    seen_message_ids = set()
    
    # Stream using stream_mode="values"
    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="values"
    ):
        if "messages" in chunk:
            for message in chunk["messages"]:
                msg_id = message.id
                
                # Skip if we've already printed this message
                if msg_id in seen_message_ids:
                    continue
                    
                seen_message_ids.add(msg_id)
                msg_type = getattr(message, 'type', 'unknown')
                
                # Format based on message type
                if msg_type == 'human':
                    content = message.content
                    if max_content_length and len(content) > max_content_length:
                        content = content[:max_content_length] + "..."
                    print(f"[USER]: {content}")
                    print("-" * 80)
                    
                elif msg_type == 'ai':
                    if hasattr(message, 'tool_calls') and message.tool_calls:
                        print(f"[AGENT] Calling {len(message.tool_calls)} tool(s):")
                        for tc in message.tool_calls:
                            tool_name = tc.get('name', 'unknown')
                            tool_args = tc.get('args', {})
                            
                            # Show detailed formatting for specific tools
                            if tool_name == 'write_todos' and show_details:
                                todos = tool_args.get('todos', [])
                                print(f"  - {tool_name}: Planning {len(todos)} tasks")
                                for i, todo in enumerate(todos, 1):
                                    status = todo.get('status', 'pending')
                                    content = todo.get('content', '')[:50]
                                    print(f"      {i}. [{status}] {content}...")
                            elif tool_name == 'internet_search' and show_details:
                                query_str = tool_args.get('query', '')
                                print(f"  - {tool_name}: '{query_str}'")
                            elif tool_name == 'write_file' and show_details:
                                path = tool_args.get('path', '')
                                print(f"  - {tool_name}: {path}")
                            else:
                                # Simple format: show function call with key args
                                args_str = ', '.join(f'{k}={v}' for k, v in list(tool_args.items())[:2])
                                print(f"  - {tool_name}({args_str})")
                                
                    elif message.content:
                        content = message.content
                        if max_content_length and len(content) > max_content_length:
                            content = content[:max_content_length] + "..."
                        print(f"[AGENT]: {content}")
                    print("-" * 80)
                    
                elif msg_type == 'tool':
                    tool_name = getattr(message, 'name', 'unknown')
                    print(f"[TOOL: {tool_name}] Completed")
                    print("-" * 80)
    
    print("\n" + "="*80)
    print("Execution completed!")
    print("="*80)

print("Helper function 'stream_agent_response' created!")

Helper function 'stream_agent_response' created!


In [9]:
# Simple query with streaming
simple_query = "What is LangGraph?"

# Use the helper function to stream and display the response
stream_agent_response(agent, simple_query, show_details=False, max_content_length=None)

Query: What is LangGraph?

Streaming Agent Actions:

[USER]: What is LangGraph?
--------------------------------------------------------------------------------
[AGENT] Calling 1 tool(s):
  - internet_search(query=What is LangGraph)
--------------------------------------------------------------------------------
[TOOL: internet_search] Completed
--------------------------------------------------------------------------------
[AGENT] Calling 1 tool(s):
  - internet_search(query=LangGraph core concepts State Nodes Edges)
--------------------------------------------------------------------------------
[TOOL: internet_search] Completed
--------------------------------------------------------------------------------
[AGENT] Calling 1 tool(s):
  - internet_search(query=LangGraph human in the loop persistence checkpointer)
--------------------------------------------------------------------------------
[TOOL: internet_search] Completed
---------------------------------------------------------

In [10]:
# Complex multi-step task with streaming
complex_query = """Research the current state of AI agents in 2026. I need:
1. The main frameworks and tools being used
2. Key differences between agent architectures
3. Real-world applications and use cases
4. Save your findings to a file called 'ai_agents_research_2025.txt' in the current directory.
"""

# Use the helper function with detailed output for complex tasks
stream_agent_response(agent, complex_query, show_details=True, max_content_length=None)

Query: Research the current state of AI agents in 2026. I need:
1. The main frameworks and tools being used...

Streaming Agent Actions:

[USER]: Research the current state of AI agents in 2026. I need:
1. The main frameworks and tools being used
2. Key differences between agent architectures
3. Real-world applications and use cases
4. Save your findings to a file called 'ai_agents_research_2025.txt' in the current directory.

--------------------------------------------------------------------------------
[AGENT] Calling 3 tool(s):
  - internet_search: 'AI agent frameworks tools 2025 2026'
  - internet_search: 'AI agent architectures differences 2025 2026'
  - internet_search: 'AI agents real world applications use cases 2025 2026'
--------------------------------------------------------------------------------
[TOOL: internet_search] Completed
--------------------------------------------------------------------------------
[TOOL: internet_search] Completed
---------------------------